In [3]:
import pandas as pd
import numpy as np

df = pd.read_csv('2019-Nov_smartphone_preprocessed_final.csv')

df.head()

,event_time,event_type,product_id,category_id,category_code,brand,price,user_id,user_session,is_price_error,hour,day_of_week,day_name,time_segment,is_outlier,is_purchase,is_cart
0,2019-11-01 09:00:00,view,1003461,-1769995873,electronics.smartphone,xiaomi,489.07,520088904,4d3b30da-a5e4-49df-b1a8-ba5943f1dd33,False,9,4,Friday,오전 (6-12시),False,0,0
1,2019-11-01 09:00:01,view,1004775,-1769995873,electronics.smartphone,xiaomi,183.27,558856683,313628f1-68b8-460d-84f6-cec7a8796ef2,False,9,4,Friday,오전 (6-12시),False,0,0
2,2019-11-01 09:00:02,view,1004258,-1769995873,electronics.smartphone,apple,732.07,532647354,d2d3d2c6-631d-489e-9fb5-06f340b85be0,False,9,4,Friday,오전 (6-12시),False,0,0
3,2019-11-01 09:00:07,view,1004566,-1769995873,electronics.smartphone,huawei,164.84,566265908,52c2c76c-b79e-4794-86ff-badc76d35f5a,False,9,4,Friday,오전 (6-12시),False,0,0
4,2019-11-01 09:00:08,view,1004708,-1769995873,electronics.smartphone,huawei,151.99,566143627,aa610ab3-5c60-4551-8a2b-8c999dddf843,False,9,4,Friday,오전 (6-12시),False,0,0


In [4]:
df = df.sort_values(['user_session', 'event_time']).copy()

In [5]:
grouped = df.groupby(['user_session', 'product_id'])

In [6]:
event_first_time = (
    df
    .groupby(['user_session', 'product_id', 'event_type'])['event_time']
    .min()
    .unstack()
    .reset_index()
)

for col in ['view', 'cart', 'purchase']:
    if col not in event_first_time.columns:
        event_first_time[col] = pd.NaT

# 존재 여부 컬럼 따로 생성
event_first_time['has_view'] = event_first_time['view'].notna()
event_first_time['has_cart'] = event_first_time['cart'].notna()
event_first_time['has_purchase'] = event_first_time['purchase'].notna()

# 순서 검증
event_first_time['view_to_cart'] = (
    event_first_time['has_view'] &
    event_first_time['has_cart'] &
    (event_first_time['view'] < event_first_time['cart'])
)

event_first_time['view_to_cart_to_purchase'] = (
    event_first_time['has_view'] &
    event_first_time['has_cart'] &
    event_first_time['has_purchase'] &
    (event_first_time['view'] < event_first_time['cart']) &
    (event_first_time['cart'] < event_first_time['purchase'])
)

funnel_df = event_first_time[[
    'user_session',
    'product_id',
    'has_view',
    'view_to_cart',
    'view_to_cart_to_purchase'
]].rename(columns={
    'has_view': 'view'
})

funnel_df['segment'] = np.select(
    [
        funnel_df['view_to_cart_to_purchase'],  # 완전 전환
        
        funnel_df['view_to_cart'] & ~funnel_df['view_to_cart_to_purchase'],  # cart까지 갔다가 이탈
        
        funnel_df['view'] & ~funnel_df['view_to_cart']  # view만 하고 이탈
    ],
    [
        'conversion',
        'view_cart_drop',
        'view_only_drop'
    ],
    default='etc'
)

In [7]:
segment_summary = (
    funnel_df['segment']
    .value_counts()
    .reset_index()
)

segment_summary.columns = ['segment', 'count']

segment_summary['ratio'] = (
    segment_summary['count'] / segment_summary['count'].sum() * 100
)

segment_summary

,segment,count,ratio
0,view_only_drop,8828187,92.208351
1,view_cart_drop,438767,4.582819
2,conversion,299642,3.129691
3,etc,7577,0.079140


In [8]:
funnel_df['segment'].value_counts(normalize=True) * 100

segment
view_only_drop    92.208351
view_cart_drop     4.582819
conversion         3.129691
etc                0.079140
Name: proportion, dtype: float64

In [9]:
funnel_summary = pd.DataFrame({
    'step': ['view', 'view_to_cart', 'view_to_cart_to_purchase'],
    'count': [
        funnel_df['view'].sum(),
        funnel_df['view_to_cart'].sum(),
        funnel_df['view_to_cart_to_purchase'].sum()
    ]
})

funnel_summary

,step,count
0,view,9566596
1,view_to_cart,738409
2,view_to_cart_to_purchase,299642


In [10]:
# 1. 구매 전환 케이스
conversion_df = funnel_df[
    funnel_df['segment'] == 'conversion'
].copy()

# 2. view만 하고 이탈한 케이스
view_only_drop_df = funnel_df[
    funnel_df['segment'] == 'view_only_drop'
].copy()

# 3. view → cart 후 구매 없이 이탈한 케이스
view_cart_drop_df = funnel_df[
    funnel_df['segment'] == 'view_cart_drop'
].copy()

In [11]:
df_segment = df.merge(
    funnel_df[['user_session', 'product_id', 'segment']],
    on=['user_session', 'product_id'],
    how='left'
)

df_segment.head()

,event_time,event_type,product_id,category_id,category_code,brand,price,user_id,user_session,is_price_error,hour,day_of_week,day_name,time_segment,is_outlier,is_purchase,is_cart,segment
0,2019-11-13 15:02:54,view,1005140,-1769995873,electronics.smartphone,apple,1422.10,457789173,0000009d-1f5b-40b9-bd23-db4f3d973ae3,False,15,2,Wednesday,오후 (12-18시),False,0,0,view_only_drop
1,2019-11-09 22:58:06,view,1004307,-1769995873,electronics.smartphone,honor,308.63,569353390,000001d5-f8f8-4e40-b8d6-224155713521,False,22,5,Saturday,저녁 (18-24시),False,0,0,view_only_drop
2,2019-11-17 14:36:28,view,1005135,-1769995873,electronics.smartphone,apple,1661.15,518709589,00000616-f016-4c01-b323-438486d9d3ee,False,14,6,Sunday,오후 (12-18시),False,0,0,view_only_drop
3,2019-11-17 14:36:38,view,1005116,-1769995873,electronics.smartphone,apple,1013.85,518709589,00000616-f016-4c01-b323-438486d9d3ee,False,14,6,Sunday,오후 (12-18시),False,0,0,view_only_drop
4,2019-11-17 14:37:50,view,1005144,-1769995873,electronics.smartphone,apple,1661.30,518709589,00000616-f016-4c01-b323-438486d9d3ee,False,14,6,Sunday,오후 (12-18시),False,0,0,view_only_drop


이탈 집단은 view 시점의 price / price 이상치 처리..

In [12]:
segment_price_detail = (
    df_segment[df_segment['event_type'] == 'view']
    .groupby('segment')
    .agg(
        view_count=('price', 'count'),
        avg_price=('price', 'mean'),
        median_price=('price', 'median'),
        min_price=('price', 'min'),
        max_price=('price', 'max')
    )
    .reset_index()
)

segment_price_detail

,segment,view_count,avg_price,median_price,min_price,max_price
0,conversion,708988,456.406365,258.57,0.0,2562.49
1,view_cart_drop,1126787,480.101775,278.69,0.0,2562.49
2,view_only_drop,12992619,487.862898,291.46,0.0,2562.49


In [13]:
conversion_purchase_price = (
    df_segment[
        (df_segment['segment'] == 'conversion') &
        (df_segment['event_type'] == 'purchase')
    ]
    .groupby('segment')
    .agg(
        purchase_count=('price', 'count'),
        avg_purchase_price=('price', 'mean'),
        median_purchase_price=('price', 'median'),
        min_purchase_price=('price', 'min'),
        max_purchase_price=('price', 'max')
    )
    .reset_index()
)

conversion_purchase_price

,segment,purchase_count,avg_purchase_price,median_purchase_price,min_purchase_price,max_purchase_price
0,conversion,323889,455.885938,255.59,35.75,2562.49


세그먼트 별 브랜드 분포

In [14]:
segment_brand = (
    df_segment[df_segment['event_type'] == 'view']
    .groupby(['segment', 'brand'])
    .size()
    .reset_index(name='count')
    .sort_values(['segment', 'count'], ascending=[True, False])
)

segment_brand.head(20)

,segment,brand,count
24,conversion,samsung,293056
0,conversion,apple,227423
32,conversion,xiaomi,100256
11,conversion,huawei,42897
21,conversion,oppo,29711
17,conversion,meizu,3817
31,conversion,vivo,3780
18,conversion,nokia,1353
9,conversion,honor,1322
20,conversion,oneplus,1118


In [15]:
segment_brand['segment_total'] = (
    segment_brand.groupby('segment')['count'].transform('sum')
)

segment_brand['ratio'] = (
    segment_brand['count'] / segment_brand['segment_total'] * 100
)

segment_brand.sort_values(['segment', 'ratio'], ascending=[True, False]).head(30)

,segment,brand,count,segment_total,ratio
24,conversion,samsung,293056,708988,41.334409
0,conversion,apple,227423,708988,32.077130
32,conversion,xiaomi,100256,708988,14.140719
11,conversion,huawei,42897,708988,6.050455
21,conversion,oppo,29711,708988,4.190621
17,conversion,meizu,3817,708988,0.538373
31,conversion,vivo,3780,708988,0.533154
18,conversion,nokia,1353,708988,0.190835
9,conversion,honor,1322,708988,0.186463
20,conversion,oneplus,1118,708988,0.157690


상위 구매 / 이탈집단 상위 조회

In [16]:
segment_product_purchase = (
    df_segment[
        (df_segment['segment'] == 'conversion') &
        (df_segment['event_type'] == 'purchase')
    ]
    .groupby(['segment', 'product_id'])
    .agg(
        revenue=('price', 'sum'),
        purchase_count=('price', 'count')
    )
    .reset_index()
    .sort_values('revenue', ascending=False)
)

segment_product_purchase.head(10)

,segment,product_id,revenue,purchase_count
651,conversion,1005115,17258899.23,18636
641,conversion,1005105,9446741.04,7001
671,conversion,1005135,5795457.42,3504
263,conversion,1004249,5557564.33,7220
22,conversion,1002544,4803580.74,9997
432,conversion,1004767,4783788.97,19404
652,conversion,1005116,4174127.16,4165
489,conversion,1004856,3568764.46,28048
11,conversion,1002524,2895375.73,5276
499,conversion,1004870,2620742.28,9207


In [17]:
segment_product_purchase_count = (
    df_segment[
        (df_segment['segment'] == 'conversion') &
        (df_segment['event_type'] == 'purchase')
    ]
    .groupby(['segment', 'product_id'])
    .agg(
        revenue=('price', 'sum'),
        purchase_count=('price', 'count')
    )
    .reset_index()
    .sort_values('purchase_count', ascending=False)
)

segment_product_purchase_count.head(10)

,segment,product_id,revenue,purchase_count
489,conversion,1004856,3568764.46,28048
432,conversion,1004767,4783788.97,19404
651,conversion,1005115,17258899.23,18636
470,conversion,1004833,2013775.61,11765
22,conversion,1002544,4803580.74,9997
499,conversion,1004870,2620742.28,9207
636,conversion,1005100,1227470.14,9039
263,conversion,1004249,5557564.33,7220
641,conversion,1005105,9446741.04,7001
473,conversion,1004836,1588923.24,6859


In [18]:
segment_product_view = (
    df_segment[
        (df_segment['segment'].isin(['view_only_drop', 'view_cart_drop'])) &
        (df_segment['event_type'] == 'view')
    ]
    .groupby(['segment', 'product_id'])
    .agg(
        view_count=('product_id', 'size'),
        avg_price=('price', 'mean'),
        median_price=('price', 'median')
    )
    .reset_index()
    .sort_values(['segment', 'view_count'], ascending=[True, False])
)

segment_product_view.head(20)

,segment,product_id,view_count,avg_price,median_price
706,view_cart_drop,1005115,65064,921.724049,914.00
540,view_cart_drop,1004856,55661,126.893965,126.18
483,view_cart_drop,1004767,52966,245.931669,243.49
550,view_cart_drop,1004870,31815,282.469488,278.69
749,view_cart_drop,1005160,26578,202.148435,202.98
292,view_cart_drop,1004249,24600,753.169653,739.04
523,view_cart_drop,1004833,23798,170.681430,168.78
691,view_cart_drop,1005100,23419,136.210590,136.76
707,view_cart_drop,1005116,21731,990.513202,979.43
696,view_cart_drop,1005105,20162,1356.181442,1364.00


세그먼트 별 조회 상품수

In [19]:
views_segment = df_segment[df_segment['event_type'] == 'view'].copy()

session_view_depth_segment = (
    views_segment.groupby(['segment', 'user_session'])
    .agg(
        view_event_count=('product_id', 'size'),
        unique_view_products=('product_id', 'nunique')
    )
    .reset_index()
)

segment_view_depth = (
    session_view_depth_segment.groupby('segment')
    .agg(
        avg_view_events_per_session=('view_event_count', 'mean'),
        median_view_events_per_session=('view_event_count', 'median'),
        avg_unique_products_per_session=('unique_view_products', 'mean'),
        median_unique_products_per_session=('unique_view_products', 'median')
    )
    .reset_index()
)

segment_view_depth

,segment,avg_view_events_per_session,median_view_events_per_session,avg_unique_products_per_session,median_unique_products_per_session
0,conversion,2.555879,2.0,1.080200,1.0
1,view_cart_drop,2.833522,2.0,1.103364,1.0
2,view_only_drop,3.403949,2.0,2.312906,1.0


세그먼트별 동일 세션 내 모델 비교 깊이

In [20]:
segment_compare_depth = (
    views_segment.groupby(['segment', 'user_session'])
    .agg(
        unique_models_compared=('product_id', 'nunique')
    )
    .reset_index()
)

segment_compare_summary = (
    segment_compare_depth.groupby('segment')
    .agg(
        avg_model_compare_depth=('unique_models_compared', 'mean'),
        median_model_compare_depth=('unique_models_compared', 'median'),
        max_model_compare_depth=('unique_models_compared', 'max')
    )
    .reset_index()
)

segment_compare_ratio = (
    segment_compare_depth.groupby('segment')
    .apply(lambda g: pd.Series({
        'compare_2plus_session_ratio': (g['unique_models_compared'] >= 2).mean() * 100,
        'compare_3plus_session_ratio': (g['unique_models_compared'] >= 3).mean() * 100
    }))
    .reset_index()
)

segment_compare_summary = segment_compare_summary.merge(
    segment_compare_ratio,
    on='segment',
    how='left'
)

segment_compare_summary

,segment,avg_model_compare_depth,median_model_compare_depth,max_model_compare_depth,compare_2plus_session_ratio,compare_3plus_session_ratio
0,conversion,1.080200,1.0,15,6.548424,1.055535
1,view_cart_drop,1.103364,1.0,30,8.437043,1.335553
2,view_only_drop,2.312906,1.0,125,42.150553,25.091769


In [21]:
final_segment_compare = (
    segment_price_detail
    .merge(segment_view_depth, on='segment', how='left')
    .merge(segment_compare_summary, on='segment', how='left')
)

final_segment_compare

,segment,view_count,avg_price,median_price,min_price,max_price,avg_view_events_per_session,median_view_events_per_session,avg_unique_products_per_session,median_unique_products_per_session,avg_model_compare_depth,median_model_compare_depth,max_model_compare_depth,compare_2plus_session_ratio,compare_3plus_session_ratio
0,conversion,708988,456.406365,258.57,0.0,2562.49,2.555879,2.0,1.080200,1.0,1.080200,1.0,15,6.548424,1.055535
1,view_cart_drop,1126787,480.101775,278.69,0.0,2562.49,2.833522,2.0,1.103364,1.0,1.103364,1.0,30,8.437043,1.335553
2,view_only_drop,12992619,487.862898,291.46,0.0,2562.49,3.403949,2.0,2.312906,1.0,2.312906,1.0,125,42.150553,25.091769
